# Module 5.3: Forecasting Consumption & Pressure Drift: ARIMA vs XGBoost
### 🇵🇱 Prognozowanie Zużycia i Dryfu Ciśnienia: ARIMA kontra XGBoost

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m05_l03_forecasting_consumption_pressure_drift.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Forecast factory compressed air and natural gas demand using statistical ARIMA and machine learning XGBoost with temporal lag features.  
> 🇵🇱 **Streszczenie:** Prognozuj zapotrzebowanie na sprężone powietrze i gaz przy użyciu statystycznego modelu ARIMA oraz algorytmu XGBoost z cechami opóźnionymi.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Plant Energy & Fluid Demand**: Forecasting natural gas and compressed air consumption prevents expensive peak-power penalty tariffs.
2. **ARIMA Modeling**: Captures linear autoregression and moving average residuals across stationary differenced data.
3. **XGBoost with Lag Features**: Non-linear gradient boosting models capture interaction effects between production schedule shifts, outside ambient temperature, and autoregressive historical lags ($t-1, t-24$).


In [ ]:
# 1. Synthesize Factory Natural Gas Consumption Log (1,000 Hours)
np.random.seed(42)
n_hours = 1000
t_hours = np.arange(n_hours)

# Diurnal shift cycle (24-hour periodicity) + day-of-week modulation
diurnal = 35.0 + 15.0 * np.sin(2 * np.pi * t_hours / 24)
ambient_temp = 18.0 - 8.0 * np.sin(2 * np.pi * t_hours / 24) + np.random.normal(0, 1.5, n_hours)

# Gas consumption: higher during cold shifts and high production
gas_m3_per_hr = diurnal + 0.8 * (25.0 - ambient_temp) + np.random.normal(0, 2.5, n_hours)

df_gas = pd.DataFrame({
    "hour": t_hours,
    "ambient_temp": ambient_temp,
    "gas_m3_hr": gas_m3_per_hr
})

# Construct autoregressive lag features for machine learning
df_gas["lag_1"]  = df_gas["gas_m3_hr"].shift(1)
df_gas["lag_24"] = df_gas["gas_m3_hr"].shift(24)
df_gas["rolling_mean_6"] = df_gas["gas_m3_hr"].rolling(6).mean().shift(1)
df_gas = df_gas.dropna()
print(f"✓ Engineered feature matrix: {len(df_gas)} hourly rows.")


In [ ]:
# 2. Train Walk-Forward Forecasting Model & Benchmark MAPE
from sklearn.ensemble import GradientBoostingRegressor

split_pt = len(df_gas) - 168  # Hold out final 7 days (168 hours) for testing
train_df = df_gas.iloc[:split_pt]
test_df  = df_gas.iloc[split_pt:]

features = ["ambient_temp", "lag_1", "lag_24", "rolling_mean_6"]
X_train, y_train = train_df[features], train_df["gas_m3_hr"]
X_test, y_test   = test_df[features], test_df["gas_m3_hr"]

# Train Gradient Boosting Regressor (XGBoost proxy)
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, max_depth=4, random_state=42)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)

# Calculate Mean Absolute Percentage Error (MAPE)
mape = np.mean(np.abs((y_test.to_numpy() - y_pred) / y_test.to_numpy())) * 100

plt.figure(figsize=(14, 5))
plt.plot(test_df["hour"].iloc[:72], y_test.iloc[:72], color='#0284c7', linewidth=2, label='Actual Gas Consumption [m³/hr]')
plt.plot(test_df["hour"].iloc[:72], y_pred[:72], color='#10b981', linestyle='--', linewidth=2, label=f'Model Forecast (MAPE = {mape:.2f}%)')
plt.fill_between(test_df["hour"].iloc[:72], y_pred[:72]*0.95, y_pred[:72]*1.05, color='#10b981', alpha=0.15, label='±5% Precision Band')

plt.title("72-Hour Multi-Step Industrial Utility Demand Forecast", fontsize=13, fontweight='bold')
plt.xlabel("Production Hour Index")
plt.ylabel("Natural Gas Flow [m³/hr]")
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Fit statistical ARIMA model on hourly factory gas consumption logs *(🇵🇱 Dopasuj model statystyczny ARIMA do godzinowych logów zużycia gazu w fabryce)*
- [ ] **Task 2:** Construct autoregressive lag features and calendar encodings for XGBoost regression *(🇵🇱 Zbuduj cechy opóźnione autoregresyjnie i kodowanie kalendarzowe dla modelu XGBoost)*
- [ ] **Task 3:** Evaluate multi-step walk-forward forecast horizon using Mean Absolute Percentage Error (MAPE) *(🇵🇱 Oceń wielokrokowy horyzont prognozy za pomocą błędu MAPE w walidacji kroczącej)*


In [ ]:
# Exercise Challenge:
# Assert that the forecasting model achieves MAPE under 8%
print(f"Achieved Forecast MAPE: {mape:.2f}%")
assert mape < 8.0, "Forecasting error exceeded acceptable industrial tolerance!"
print("✓ Consumption forecasting test verified.")
